In [1]:
import sys
import os
import numpy as np
import xarray as xr
import pandas as pd
import datetime

import matplotlib as mpl
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [ ]:
xls = []
lon_bnds, lat_bnds = (8, 32), (12,-15)
folder = '/Volumes/ESA_F4R/tropomi/tropomi2/'
#folder = '/Users/ellendyer/Desktop/tropomi/'
arr = os.listdir(folder)
print(arr)
for I,F in enumerate(arr):
    try:
        x = xr.open_dataset(folder+F,
                            group="PRODUCT",
                            drop_variables=['level','corner','glintflag','ground_pixel','nwin',
                                            'hdo_column_precision','hdo_column_apriori',
                                            'hdo_profile_apriori','h2o_profile_apriori',
                                            'delta_time','layer',
                                            'h2o_column_precision','h2o_column_apriori'])
        x = x.rename({'latitude':'lat','longitude':'lon'})
        x = x.assign_coords({"ground_pixel": x.ground_pixel.values})
        x = x.where((x.lat < 12.) & (x.lat > 5.),drop=True)
        x = x.where((x.lon < 32.) & (x.lon > 10.),drop=True)
        xls.append(x)
        x.close()
        print(F)
    except:
        print('not nc4: ',F)

['S5P_PAL__L2__HDO__S_20180629T132804_20180629T150934_03677_01_100300_20250307T191718.nc', 'S5P_PAL__L2__HDO__S_20190126T121926_20190126T140056_06670_01_100300_20250309T215830.nc', 'S5P_PAL__L2__HDO__S_20180511T120444_20180511T134614_02981_01_100300_20250308T172115.nc', 'S5P_PAL__L2__HDO__S_20181220T121144_20181220T135314_06145_01_100300_20250308T101916.nc', 'S5P_PAL__L2__HDO__S_20180724T121731_20180724T135901_04031_01_100300_20250308T223742.nc', 'S5P_PAL__L2__HDO__S_20180510T122342_20180510T140512_02967_01_100300_20250308T173250.nc', 'S5P_PAL__L2__HDO__S_20180502T131356_20180502T145526_02854_01_100300_20250308T160257.nc', 'S5P_PAL__L2__HDO__S_20181107T103848_20181107T122018_05534_01_100300_20250308T065844.nc', 'S5P_PAL__L2__HDO__S_20180517T133351_20180517T151521_03067_01_100300_20250308T191341.nc', 'S5P_PAL__L2__HDO__S_20180617T084855_20180617T103025_03504_01_100300_20250307T203912.nc', 'S5P_PAL__L2__HDO__S_20180611T122436_20180611T140606_03421_01_100300_20250307T184308.nc', 'S5P_PAL_

: 

In [ ]:
trp = xr.concat(xls,dim='time')
print(trp)


In [ ]:
trp.to_netcdf(folder+"TROPOMI_merged.nc")

In [ ]:
#Plot climatology

tpN = trp.where((trp.lat < 12.) & (trp.lat > 5.),drop=True)
tpN = tpN.mean(dim=('scanline','ground_pixel'))
print(tpN)
tpN = tpN.groupby('time.dayofyear').mean('time')
tpN['deltad'].plot()
plt.title('North TROPOMI')
#plt.ylim(0.2,0.85)
plt.show()
plt.clf()

tpE = trp.where((trp.lat < 5.) & (trp.lat > -5.),drop=True)
tpE = tpE.mean(dim=('scanline','ground_pixel'))
tpE = tpE.groupby('time.dayofyear').mean('time')
tpE['deltad'].plot()
plt.title('Equatorial TROPOMI')
#plt.ylim(0.2,0.85)
plt.show()
plt.clf()

tpS = trp.where((trp.lat < -5.) & (trp.lat > -15.),drop=True)
tpS = tpS.mean(dim=('scanline','ground_pixel'))
tpS = tpS.groupby('time.dayofyear').mean('time')
tpS['deltad'].plot()
plt.title('South TROPOMI')
#plt.ylim(0.2,0.85)
plt.show()
plt.clf()